# Laboratorio: Búsqueda en entornos complejos

- Bryan Alberto Martínez Orellana - 23542
- Adriana Sophia Palacios Contreras - 23044

## Búsqueda Local 

En este laboratorio se estudiarán algoritmos de búsqueda local y heurística en un entorno de espacio de estados complejo. El problema elegido es el de las 8 reinas, en el cual se deben ubicar 8 reinas sobre un tablero de ajedrez de 8x8 de manera que ninguna reina pueda atacar a otra. 

A diferencia de los métodos de búsqueda sistemática, los algoritmos de búsqueda local no construyen explícitamente un árbol completo, sino que exploran el espacio de estados moviéndose entre configuraciones vecinas. 

## Descripción del problema

El problema de las 8 reinas consiste en colocar 8 reinas sobre un tablero de 8x8 de forma que:
- no compartan la misma fila
- no compartan la misma columna
- no compartan la misma diagonal.

*Representación del estado*

Cada estado se representará como un arreglo de longitud 8:



In [1]:
estado = [0, 4, 7, 5, 2, 6, 1, 3]

significa:
- en la columna 0 hay una reina en la fila 0,
- en la columna 1 hay una reina en la fila 4,
- ...
- en la columna 7 hay una reina en la fila 3.

## Implementación

*En parejas*, deberán implementar: 
- Hill Climbing
- Una variación del Hill Climbing (Stochastic/ First-choice/ Random-restart)
- Beam Search con beam width = _k_

Cada grupo debe realizar experimentos comparativos entre los algoritmos implementados.

### Experimentos requeridos:

Ejecutar cada algoritmo 1000 veces con estados iniciales aleatorios y reportar:

- porcentaje de éxito,
- promedio de largo de episodio (hasta éxito o estancarse)
- promedio de valor heurístico final,
- tiempo promedio de ejecución.

Para Beam Search, repetir los experimentos con distintos valores de _k_ (2,5,10). 

## Discuta 

¿Qué tan frecuentemente Hill Climbing encuentra una solución?

¿Qué tipo de problemas presenta Hill Climbing en este dominio?

¿La variante elegida mejora el desempeño? ¿Por qué?

¿Cómo afecta el valor de k en Beam Search?

¿Cuál algoritmo resultó más efectivo?

¿Qué relación existe entre costo computacional y tasa de éxito?

## Funciones útiles

In [2]:
def es_solucion(estado):
    """
    Verifica si un estado representa una solución válida al problema
    de las 8 reinas.

    Parámetros:
        estado (list): lista de 8 enteros, donde el índice representa
                       la columna y el valor representa la fila.

    Retorna:
        bool: True si es una solución válida, False en caso contrario.
    """
    if not isinstance(estado, list) or len(estado) != 8:
        return False

    # Verificar que todas las filas sean enteros entre 0 y 7
    for fila in estado:
        if not isinstance(fila, int) or fila < 0 or fila > 7:
            return False

    n = 8

    for c1 in range(n):
        for c2 in range(c1 + 1, n):
            r1 = estado[c1]
            r2 = estado[c2]

            # Misma fila
            if r1 == r2:
                return False

            # Misma diagonal
            if abs(r1 - r2) == abs(c1 - c2):
                return False

    return True

print(es_solucion([0, 4, 7, 5, 2, 6, 1, 3]))  # True
print(es_solucion([0, 1, 2, 3, 4, 5, 6, 7]))  # False


def heuristica(estado):
    """
    Calcula el número de pares de reinas que se atacan entre sí.
    Un estado solución tiene heurística 0.
    """
    conflictos = 0
    n = 8

    for c1 in range(n):
        for c2 in range(c1 + 1, n):
            r1 = estado[c1]
            r2 = estado[c2]

            if r1 == r2 or abs(r1 - r2) == abs(c1 - c2):
                conflictos += 1

    return conflictos

True
False


# **Implementación de Algoritmos**

## **Hill Climbing**

In [ ]:
import pandas as pd
import random
import time

# El índice representa la columna, y el valor, la fila de la reina
def generarEstadoAleatorio():
    estado = []
    for i in range(8):
        numero = random.randint(0, 7)
        estado.append(numero)
    return estado

def generarVecinos(estado):
    posiblesVecinos = []

    for columna in range(8):
        filaActual = estado[columna]
        for nuevaFila in range(8):
            if nuevaFila != filaActual:
                vecino = estado[:]
                vecino[columna] = nuevaFila
                posiblesVecinos.append(vecino)

    return posiblesVecinos

# Hill Climbing
def hillClimbing():
    estadoActual = generarEstadoAleatorio()
    cantidadPasos = 0

    while True:
        heuristicaActual = heuristica(estadoActual)

        # Ya encontramos una solucion
        if heuristicaActual == 0:
            return {
                "estadoFinal": estadoActual,
                "cantidadPasos": cantidadPasos,
                "exito": True,
                "heuristicaFinal": heuristicaActual
            }

        vecinos = generarVecinos(estadoActual)

        mejorVecino = None
        mejorHeuristica = float("inf")

        for vecino in vecinos:
            heuristicaVecino = heuristica(vecino)
            if heuristicaVecino < mejorHeuristica:
                mejorHeuristica = heuristicaVecino
                mejorVecino = vecino

        # Si ya no hay mejora, nos quedamos estancados
        if mejorHeuristica >= heuristicaActual:
            return {
                "estadoFinal": estadoActual,
                "exito": False,
                "cantidadPasos": cantidadPasos,
                "heuristicaFinal": heuristicaActual
            }

        estadoActual = mejorVecino
        cantidadPasos += 1

def experimentoHillClimbing():
    exitos = 0
    sumaPasos = 0
    sumaHeuristicaFinal = 0
    sumaTiempo = 0.0

    for i in range(1000):
        inicio = time.perf_counter()
        resultado = hillClimbing()
        fin = time.perf_counter()

        if resultado["exito"]:
            exitos += 1

        sumaPasos += resultado["cantidadPasos"]
        sumaHeuristicaFinal += resultado["heuristicaFinal"]
        sumaTiempo += (fin - inicio)

    estadisticas = {
        "porcentajeExito": (exitos / 1000) * 100,
        "promedioLargoEpisodio": sumaPasos / 1000,
        "promedioHeuristicaFinal": sumaHeuristicaFinal / 1000,
        "tiempoPromedioSegundos": sumaTiempo / 1000
    }

    return estadisticas

estadisticasHillClimbing = experimentoHillClimbing()

dfHillClimbing = pd.DataFrame(
    [estadisticasHillClimbing],
    index=["Hill Climbing"]
)

dfHillClimbing

,porcentajeExito,promedioLargoEpisodio,promedioHeuristicaFinal,tiempoPromedioSegundos
Hill Climbing,17.2,3.264,1.199,0.000423


---

## **Random-Restart Hill Climbing**

In [ ]:
def randomRestartHillClimbing():
    cantidadPasosTotales = 0
    mejorResultado = None
    maximosReinicios = 30

    for reinicio in range(maximosReinicios + 1):
        resultado = hillClimbing()

        cantidadPasosTotales += resultado["cantidadPasos"]

        if mejorResultado is None or resultado["heuristicaFinal"] < mejorResultado["heuristicaFinal"]:
            mejorResultado = resultado

        if resultado["exito"]:
            return {
                "estadoFinal": resultado["estadoFinal"],
                "cantidadPasos": cantidadPasosTotales,
                "exito": True,
                "heuristicaFinal": resultado["heuristicaFinal"],
                "cantidadReinicios": reinicio
            }

    return {
        "estadoFinal": mejorResultado["estadoFinal"],
        "cantidadPasos": cantidadPasosTotales,
        "exito": False,
        "heuristicaFinal": mejorResultado["heuristicaFinal"],
        "cantidadReinicios": maximosReinicios
    }


def ejecutarExperimentosRandomRestartHillClimbing():
    exitos = 0
    sumaPasos = 0
    sumaHeuristicaFinal = 0
    sumaTiempo = 0.0

    for i in range(1000):
        inicio = time.perf_counter()
        resultado = randomRestartHillClimbing()
        fin = time.perf_counter()

        if resultado["exito"]:
            exitos += 1

        sumaPasos += resultado["cantidadPasos"]
        sumaHeuristicaFinal += resultado["heuristicaFinal"]
        sumaTiempo += (fin - inicio)

    estadisticas = {
        "porcentajeExito": (exitos / 1000) * 100,
        "promedioLargoEpisodio": sumaPasos / 1000,
        "promedioHeuristicaFinal": sumaHeuristicaFinal / 1000,
        "tiempoPromedioSegundos": sumaTiempo / 1000
    }

    return estadisticas


estadisticasRandomRestart = ejecutarExperimentosRandomRestartHillClimbing()

dfRandomRestart = pd.DataFrame(
    [estadisticasRandomRestart],
    index=["Random-Restart Hill Climbing"]
)

dfRandomRestart

,porcentajeExito,promedioLargoEpisodio,promedioHeuristicaFinal,tiempoPromedioSegundos
Random-Restart Hill Climbing,99.0,20.665,0.01,0.002823


---

## **Beam Search**

In [12]:
def beamSearch(k):
    maximasIteraciones = 100
    estadosActuales = []

    for i in range(k):
        estadosActuales.append(generarEstadoAleatorio())

    cantidadPasos = 0
    mejorEstado = min(estadosActuales, key=heuristica)
    mejorHeuristica = heuristica(mejorEstado)

    while cantidadPasos < maximasIteraciones:
        for estado in estadosActuales:
            heuristicaEstado = heuristica(estado)

            if heuristicaEstado < mejorHeuristica:
                mejorEstado = estado
                mejorHeuristica = heuristicaEstado

            if heuristicaEstado == 0:
                return {
                    "estadoFinal": estado,
                    "cantidadPasos": cantidadPasos,
                    "exito": True,
                    "heuristicaFinal": 0
                }

        todosLosVecinos = []

        for estado in estadosActuales:
            vecinos = generarVecinos(estado)
            todosLosVecinos.extend(vecinos)

        todosLosVecinos.sort(key=heuristica)
        mejoresVecinos = todosLosVecinos[:k]

        mejorHeuristicaNueva = heuristica(mejoresVecinos[0])

        if mejorHeuristicaNueva > mejorHeuristica:
            return {
                "estadoFinal": mejorEstado,
                "cantidadPasos": cantidadPasos,
                "exito": False,
                "heuristicaFinal": mejorHeuristica
            }

        estadosActuales = mejoresVecinos
        mejorEstado = estadosActuales[0]
        mejorHeuristica = heuristica(mejorEstado)
        cantidadPasos += 1

    return {
        "estadoFinal": mejorEstado,
        "cantidadPasos": cantidadPasos,
        "exito": False,
        "heuristicaFinal": mejorHeuristica
    }


def ejecutarExperimentosBeamSearch(k):
    exitos = 0
    sumaPasos = 0
    sumaHeuristicaFinal = 0
    sumaTiempo = 0.0

    for i in range(1000):
        inicio = time.perf_counter()
        resultado = beamSearch(k)
        fin = time.perf_counter()

        if resultado["exito"]:
            exitos += 1

        sumaPasos += resultado["cantidadPasos"]
        sumaHeuristicaFinal += resultado["heuristicaFinal"]
        sumaTiempo += (fin - inicio)

    estadisticas = {
        "porcentajeExito": (exitos / 1000) * 100,
        "promedioLargoEpisodio": sumaPasos / 1000,
        "promedioHeuristicaFinal": sumaHeuristicaFinal / 1000,
        "tiempoPromedioSegundos": sumaTiempo / 1000
    }

    return estadisticas


estadisticasBeam2 = ejecutarExperimentosBeamSearch(2)
estadisticasBeam5 = ejecutarExperimentosBeamSearch(5)
estadisticasBeam10 = ejecutarExperimentosBeamSearch(10)

dfBeamSearch = pd.DataFrame(
    [estadisticasBeam2, estadisticasBeam5, estadisticasBeam10],
    index=["Beam Search (k=2)", "Beam Search (k=5)", "Beam Search (k=10)"]
)

dfBeamSearch

,porcentajeExito,promedioLargoEpisodio,promedioHeuristicaFinal,tiempoPromedioSegundos
Beam Search (k=2),44.5,56.820,0.623,0.012102
Beam Search (k=5),73.9,29.004,0.268,0.015622
Beam Search (k=10),86.9,16.466,0.131,0.017736


---

# **Discusión**

- ¿Qué tan frecuentemente Hill Climbing encuentra una solución?

El Hill Climbing tuvo un porcentaje de éxito del 17.2%, esto quiere decir que de las 1000 pruebas realizadas, se encontró una solución en 172 de ellas. Esto tiene sentido porque en el Hill Climbing dependemos bastante del estado inicial, y de acuerdo con este es que se encuentra la solución. Dependiendo del estado, puede ocurrir que el algoritmo posteriormente se quede estancado, porque todos sus vecinos no presentan una mejor heurística que la ya encontrada.

- ¿Qué tipo de problemas presenta Hill Climbing en este dominio?

En el Hill Climbing se tienen problemas de estancamiento, porque en la variante base, si llegamos a un punto donde todos los vecinos no tienen una heurística mejor que la que ya se tenía, el algoritmo se detiene sin encontrar una solución, y no se hace algo distinto para tratar de revertir o aumentar las posibilidades de encontrar una solución, como en la variante de random-restart o beam search.

- ¿La variante elegida mejora el desempeño? ¿Por qué?

Sí, porque con la variante de random-restart esta métrica fue de 99.0%, mientras que la versión default solo tuvo 17.2%. Sin embargo, esto también implicó un mayor costo computacional y tiempo de ejecución. El Hill Climbing normal tiene un tiempo promedio en segundos de 0.000423, mientras que el random-restart tiene un tiempo promedio en segundos de 0.002823.

- ¿Cómo afecta el valor de k en Beam Search?

El valor de k en Beam Search determina la cantidad de estados que se trabajan de forma simultánea; por ejemplo, cuando comienza el algoritmo, si este es 3 evaluamos 3 estados a la vez y luego evaluamos los vecinos de esos 3 estados y seleccionamos los estados con mejor heurística. El aumento en k, al estar evaluando más posibilidades a la vez, aumenta la probabilidad de encontrar una solución óptima, pero también incrementa el costo computacional. Por ejemplo, con k=2 el porcentaje de éxito fue de 44.5%, con k=5 fue de 73.9% y con k=10 fue de 86.9%. Además, los tiempos promedio en segundos fueron de 0.012102, 0.015622 y 0.017736 respectivamente.

- ¿Cuál algoritmo resultó más efectivo?

El algoritmo que resultó más efectivo fue el Random Restart Hill Climbing, con una tasa de éxito del 99.0%. El segundo mejor fue el de Beam Search con k=10, con un porcentaje de éxito de 86.9%. Esto puede deberse a que es más beneficioso tener la oportunidad de revertir el proceso y seguir intentando, que probar múltiples caminos a la vez.

- ¿Qué relación existe entre costo computacional y tasa de éxito?

El costo computacional se refiere a los recursos que consumen los algoritmos al estarse ejecutando, por ejemplo, la memoria que se ocupa y el uso que se le da a la CPU para poder completar las tareas. En este laboratorio esto lo podemos ver reflejado en el promedio de largo de episodio y en el tiempo promedio en segundos. Por otro lado, la tasa de éxito se refiere directamente a la probabilidad de encontrar una solución óptima. Para este caso, a mayor tasa de éxito, mayor costo en tiempo y largo de episodio. Por ejemplo, el random-restart logra 99.0% de éxito, pero tarda más que Hill Climbing, y en Beam Search, al ir aumentando el valor de k, sube el éxito, pero también va aumentando el tiempo promedio de ejecución. 

---